In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Tue_Oct_29_23:50:19_PDT_2024
Cuda compilation tools, release 12.6, V12.6.85
Build cuda_12.6.r12.6/compiler.35059454_0


In [5]:
# !pip install uv
!pip install transformers
!pip install av
!pip install imageio
!pip install decord
!pip install opencv-python
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install flash-attn --no-build-isolation --no-cache-dir
!pip install pandas
!pip install kaggle
!pip install timm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 38.8 MB/s eta 0:00:00a 0:00:01


In [5]:
!rm -rf ~/.kaggle
!mkdir -p ~/.kaggle
!echo '{"username":"ohonsam","key":"KGAT_0fe505192c10984621fd10ed3f386f3c"}' > ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download ohonsam/sutd-traffic-video-qa
!mkdir -p /kaggle/input/sutd-traffic-video-qa
!unzip sutd-traffic-video-qa.zip -d /kaggle/input/sutd-traffic-video-qa

Dataset URL: https://www.kaggle.com/datasets/ohonsam/sutd-traffic-video-qa
License(s): CC0-1.0
100%|█████████████████████████████████████▉| 37.2G/37.2G [18:12<00:00, 36.2MB/s]
100%|██████████████████████████████████████| 37.2G/37.2G [18:12<00:00, 36.5MB/s]
Archive:  sutd-traffic-video-qa.zip
  inflating: /kaggle/input/sutd-traffic-video-qa/configs/configs/config.yaml  
  inflating: /kaggle/input/sutd-traffic-video-qa/questions/questions/R2_all.jsonl  
  inflating: /kaggle/input/sutd-traffic-video-qa/questions/questions/R2_test.jsonl  
  inflating: /kaggle/input/sutd-traffic-video-qa/questions/questions/R2_train.jsonl  
  inflating: /kaggle/input/sutd-traffic-video-qa/questions/questions/R3_all.jsonl  
  inflating: /kaggle/input/sutd-traffic-video-qa/questions/questions/R3_test.jsonl  
  inflating: /kaggle/input/sutd-traffic-video-qa/questions/questions/R3_train.jsonl  
  inflating: /kaggle/input/sutd-traffic-video-qa/videos_obj_tracking/b_114411w739_clip_006.mp4  
  inflating: /kaggle/

In [6]:
!kaggle config view

Configuration values from /root/.kaggle
- username: ohonsam
- auth_method: LEGACY_API_KEY
- path: None
- proxy: None
- competition: None


In [3]:
%%writefile setup_paths.py
import os, glob

DATA_ROOT = "/kaggle/input/sutd-traffic-video-qa"
VIDEO_DIR = f"{DATA_ROOT}/videos_obj_tracking"
QUESTIONS_DIR = f"{DATA_ROOT}/questions/questions"

print("VIDEO_DIR:", VIDEO_DIR, "exists:", os.path.isdir(VIDEO_DIR))
print("QUESTIONS_DIR:", QUESTIONS_DIR, "exists:", os.path.isdir(QUESTIONS_DIR))

# Find question json(s)
question_jsons = sorted(glob.glob(os.path.join(QUESTIONS_DIR, "**", "*.jsonl"), recursive=True))
print(f"Found {len(question_jsons)} question JSON file(s):")
for p in question_jsons[:50]:
    print(" -", p)
if len(question_jsons) > 50:
    print(" ...")

# Pick split
MODE = "test"  # change to "train" / "val" if needed
def pick_questions_path(split: str) -> str:
    split = (split or "").lower()
    for p in question_jsons:
        if split and split in os.path.basename(p).lower():
            return p
    return question_jsons[0] if question_jsons else ""

QUESTIONS_PATH = pick_questions_path(MODE)
print("QUESTIONS_PATH:", QUESTIONS_PATH)

OUTPUT_DIR = "/kaggle/working/output"
KEYFRAME_DIR = "/kaggle/temp/keyframes"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(KEYFRAME_DIR, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)
print("KEYFRAME_DIR:", KEYFRAME_DIR)

Overwriting setup_paths.py


In [ ]:
# Create sutd_train.yaml using paths from setup_paths.py
from setup_paths import DATA_ROOT, VIDEO_DIR, OUTPUT_DIR
import os

# Define paths for training config
TRAIN_JSON_PATH = os.path.join(DATA_ROOT, "sutd_llava_train.json")  # Adjust filename if different
VIDEO_ROOT = VIDEO_DIR

# Create the YAML content
yaml_content = f"""datasets:
  - json_path: {TRAIN_JSON_PATH}
    data_root: {VIDEO_ROOT}/
    media_type: video
    sampling_strategy: all
"""

# Write to sutd_train.yaml
yaml_path = os.path.join(OUTPUT_DIR, "sutd_train.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"Created sutd_train.yaml at: {yaml_path}")
print("\nContent:")
print(yaml_content)

In [ ]:
%%writefile videochat_sutd.py
import torch
import json
import os
import pandas as pd
from tqdm import tqdm
from torchvision import transforms as T
from transformers import AutoModel, AutoTokenizer
from setup_paths import VIDEO_DIR, QUESTIONS_PATH, OUTPUT_DIR

# model setting
model_path = 'OpenGVLab/VideoChat-Flash-Qwen2_5-2B_res448'

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModel.from_pretrained(model_path, trust_remote_code=True).to(torch.bfloat16).cuda()
image_processor = model.get_vision_tower().image_processor
print(f"Attention: f{model.config._attn_implementation}")

# Print VRAM usage after loading model
print(f"VRAM after loading model: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"VRAM reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

mm_llm_compress = False # use the global compress or not
if mm_llm_compress:
    model.config.mm_llm_compress = True
    model.config.llm_compress_type = "uniform0_attention"
    model.config.llm_compress_layer_list = [4, 18]
    model.config.llm_image_token_ratio_list = [1, 0.75, 0.25]
else:
    model.config.mm_llm_compress = False

# evaluation setting
max_num_frames = 512
generation_config = dict(
    do_sample=True,
    temperature=0.1,
    max_new_tokens=256,
    top_p=0.1,
    num_beams=1
)

# Load test questions (JSONL format)
test_jsonl_path = QUESTIONS_PATH
video_base_path = VIDEO_DIR
output_json_path = os.path.join(OUTPUT_DIR, "sutd_test_with_answers_video_chat.json")

print(f"Loading questions from {test_jsonl_path}")

# Parse JSONL file
test_data = []
with open(test_jsonl_path, "r") as f:
    for line_num, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        # Skip header row
        if line_num == 0 and row[0] == "record_id":
            continue
        # Parse row: [record_id, vid_id, vid_filename, perspective, q_body, option0, option1, option2, option3, answer]
        test_data.append({
            "record_id": row[0],
            "vid_id": row[1],
            "vid_filename": row[2],
            "perspective": row[3],
            "question": row[4],
            "question_type": row[5],
            "options": [row[6], row[7], row[8], row[9]],
            "answer": row[10]
        })

print(f"Total questions: {len(test_data)}")

def format_mcq_prompt(question, options, question_type):
    """Format a multiple choice question with options."""
    prompt = f"""
    You are an expert traffic safety analyst and intelligent video reasoning assistant. The images are ordered in time (earliest to latest). You must determine the correct answer by applying one of six specific reasoning capabilities:

    TASK TYPES:
    1) Basic Understanding(U): Perceive and recognize basic object attributes, road environments, and traffic states.
    - Identify road type/scene (lanes, markings, signs, intersection), agents (cars/peds/bikes), counts, positions.
    - Confirm any queried attribute by checking multiple frames.

    2) Attribution(A): Identify the causes, types, and locations of traffic events or accidents.
    - Find the earliest mistake/trigger that plausibly causes the outcome (illegal lane change, tailgating, sudden brake, red-light, obstruction, etc.).
    - Choose the most direct cause supported by the sequence, not generic priors.

    3) Event Forecasting(F): Predict future events, impending collisions, or potential risks based on current dynamics track lines.
    - Use trajectories: relative speed, closing distance, lane alignment, right-of-way, available space.
    - Decide if collision/near-miss/merge/brake is likely given motion trends.

    4) Reverse Reasoning(R): Infer the state of traffic or the sequence of events that occurred before the observed situation (antecedents).
    - Infer prior events from current state (vehicle stopped position, damage, skid/avoidance, unusual lane position).
    - Choose the option that best explains how the scene got into the observed state.

    5) Counterfactual Inference(C): Reason about hypothetical "what-if" scenarios to determine if an outcome would change under different conditions.
    - Treat the "if" condition as a change to the scene; mentally simulate the most likely outcome under traffic rules/physics.
    - Decide whether the hypothetical would remove the cause, add space/time, or still lead to the event.

    6) Introspection(I): Evaluate preventive measures and determining if specific actions or infrastructure changes could have avoided the accident.
    - Pick the safest feasible preventive action BEFORE the critical moment (slow down, keep lane, increase distance, yield, earlier braking, etc.).
    - Prefer actions that directly interrupt the causal chain seen in the video.

    EVIDENCE RULES (important):
    - Do not guess or assume events that are not clearly visible. Use ONLY visual evidence from the frame sequence + basic traffic rules/physics.
    - If a choice claims "there is no X" / "no accident" / "no barrier", verify presence/absence across ALL frames.
    - Prefer the option most consistent with the full temporal sequence (not a single frame).
    - Always ground your choice in specific visual cues (positions, speeds, signals, distances, trajectories).

    Now answer:

    Question Type: {question_type}

    Question: {question}

    Choices:
    """
    
    option_labels = ['A', 'B', 'C', 'D']
    for i, opt in enumerate(options):
        if opt:  # Only include non-empty options
            prompt += f"{option_labels[i]}. {opt}\n"

    prompt += "\nPlease provide the letter (A, B, C, or D) corresponding to your answer."
    return prompt

def parse_model_answer(output, options):
    """Parse model output to extract the selected option index."""
    output_upper = output.upper().strip()
    
    # Check for direct letter answer
    for i, letter in enumerate(['A', 'B', 'C', 'D']):
        if output_upper.startswith(letter) or f"ANSWER IS {letter}" in output_upper or f"ANSWER: {letter}" in output_upper:
            return i
    
    # Check if output contains the option text
    for i, opt in enumerate(options):
        if opt and opt.lower() in output.lower():
            return i
    
    return -1  # Could not parse

# Process each question
results = []
correct = 0
total = 0

for idx, item in enumerate(tqdm(test_data, desc="Processing videos")):
    try:
        # Construct video path
        vid_filename = item["vid_filename"]
        video_path = os.path.join(video_base_path, vid_filename)
        
        # Check if video exists
        if not os.path.exists(video_path):
            print(f"Warning: Video not found: {video_path}")
            item["model_answer"] = "ERROR: Video not found"
            item["model_answer_idx"] = -1
            results.append(item)
            continue
        
        # Format question with options
        question = format_mcq_prompt(item["question"], item["options"], item["question_type"])
        
        if idx < 3:  # Print first few for debugging
            print(f"\nProcessing {vid_filename}")
            print(f"Question: {question[:200]}...")
        
        # Run inference
        output, _ = model.chat(
            video_path=video_path, 
            tokenizer=tokenizer, 
            user_prompt=question, 
            return_history=True, 
            max_num_frames=max_num_frames, 
            generation_config=generation_config
        )
        
        # Parse answer
        predicted_idx = parse_model_answer(output, item["options"])
        
        # Add model answer to the item
        item["model_output"] = output
        item["model_answer_idx"] = predicted_idx
        item["is_correct"] = (predicted_idx == item["answer"])
        results.append(item)
        
        # Update accuracy
        total += 1
        if item["is_correct"]:
            correct += 1
        
        # Print progress
        if (idx + 1) % 50 == 0:
            print(f"\nAccuracy at {idx + 1}: {correct}/{total} = {correct/total*100:.2f}%")
        
        # Clear cache periodically
        if (idx + 1) % 10 == 0:
            torch.cuda.empty_cache()
        
    except Exception as e:
        print(f"\nError processing {item.get('vid_filename', 'unknown')}: {str(e)}")
        item["model_output"] = f"ERROR: {str(e)}"
        item["model_answer_idx"] = -1
        item["is_correct"] = False
        results.append(item)

# Calculate final accuracy
final_accuracy = correct / total * 100 if total > 0 else 0

# Save results
print(f"\nSaving results to {output_json_path}")
# with open(output_json_path, "w") as f:
#     json.dump({
#         "accuracy": final_accuracy,
#         "correct": correct,
#         "total": total,
#         "results": results
#     }, f, indent=2, ensure_ascii=False)

# Save results as csv for easier analysis
df_results = pd.DataFrame(results)
# Keep record_id,vid_filename,answer -> id,filename,answer
df_results = df_results[["record_id", "vid_filename", "model_answer_idx", "answer", "is_correct"]]
# Rename columns
df_results.columns = ["id", "filename", "answer", "gt_answer", "is_correct"]
df_results.to_csv(output_json_path.replace(".json", ".csv"), index=False)

# Print final statistics
print(f"\n{'='*50}")
print(f"Final Results:")
print(f"Accuracy: {correct}/{total} = {final_accuracy:.2f}%")
print(f"VRAM usage: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Peak VRAM usage: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print(f"Results saved to {output_json_path}")

Overwriting videochat_sutd.py


In [ ]:
# # Replace the model loading section in videochat_sutd.py

# %%writefile videochat_sutd.py
# import torch
# import json
# import os
# import pandas as pd
# from tqdm import tqdm
# from transformers import AutoModel, AutoTokenizer
# from peft import PeftModel
# from setup_paths import VIDEO_DIR, QUESTIONS_PATH, OUTPUT_DIR

# # =============================================================================
# # MODEL CONFIGURATION
# # =============================================================================

# # Base model path
# BASE_MODEL_PATH = "OpenGVLab/VideoChat-Flash-Qwen2_5-2B_res448"

# # LoRA adapter path (set to None to use base model only)
# LORA_ADAPTER_PATH = "/workspace/finetune/checkpoints/videochat-2b-sutd-lora"

# # =============================================================================
# # LOAD MODEL
# # =============================================================================

# print("Loading tokenizer...")
# tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True)

# print(f"Loading base model: {BASE_MODEL_PATH}")
# model = AutoModel.from_pretrained(
#     BASE_MODEL_PATH,
#     trust_remote_code=True,
#     torch_dtype=torch.bfloat16,
#     device_map="cuda"
# )

# # Load LoRA adapters if specified
# if LORA_ADAPTER_PATH and os.path.exists(LORA_ADAPTER_PATH):
#     print(f"Loading LoRA adapters from: {LORA_ADAPTER_PATH}")
#     model = PeftModel.from_pretrained(
#         model,
#         LORA_ADAPTER_PATH,
#         torch_dtype=torch.bfloat16,
#     )
#     # Optionally merge LoRA weights for faster inference
#     # model = model.merge_and_unload()
#     print("✅ LoRA adapters loaded successfully!")
# else:
#     print("⚠️ No LoRA adapters found, using base model only")

# model = model.eval()
# print(f"Attention implementation: {model.config._attn_implementation}")

# # Print VRAM usage
# print(f"VRAM after loading: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
# print(f"VRAM reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

# # =============================================================================
# # MODEL SETTINGS
# # =============================================================================

# mm_llm_compress = False
# if mm_llm_compress:
#     model.config.mm_llm_compress = True
#     model.config.llm_compress_type = "uniform0_attention"
#     model.config.llm_compress_layer_list = [4, 18]
#     model.config.llm_image_token_ratio_list = [1, 0.75, 0.25]
# else:
#     model.config.mm_llm_compress = False

# # =============================================================================
# # INFERENCE SETTINGS
# # =============================================================================

# max_num_frames = 512
# generation_config = dict(
#     do_sample=False,  # Deterministic for evaluation
#     temperature=0.1,
#     max_new_tokens=32,  # Short response (just the number)
#     top_p=0.1,
#     num_beams=1
# )

# # =============================================================================
# # LOAD TEST DATA
# # =============================================================================

# test_jsonl_path = QUESTIONS_PATH
# video_base_path = VIDEO_DIR
# output_json_path = os.path.join(OUTPUT_DIR, "sutd_test_finetuned.csv")

# print(f"Loading questions from {test_jsonl_path}")

# test_data = []
# with open(test_jsonl_path, "r") as f:
#     for line_num, line in enumerate(f):
#         line = line.strip()
#         if not line:
#             continue
#         row = json.loads(line)
#         if line_num == 0 and row[0] == "record_id":
#             continue
#         test_data.append({
#             "record_id": row[0],
#             "vid_id": row[1],
#             "vid_filename": row[2],
#             "perspective": row[3],
#             "question": row[4],
#             "question_type": row[5],
#             "options": [row[6], row[7], row[8], row[9]],
#             "answer": row[10]
#         })

# print(f"Total questions: {len(test_data)}")

# # =============================================================================
# # PROMPT FORMATTING (MUST MATCH TRAINING FORMAT!)
# # =============================================================================

# def format_mcq_prompt(question: str, options: list, question_type: str) -> str:
#     """Format MCQ prompt - MUST match training format exactly."""
#     choices_text = "\n".join([f"{i}. {opt}" for i, opt in enumerate(options)])
#     num_choices = len(options)
    
#     task_hints = {
#         "U": "Focus on identifying objects, road features, and traffic states visible in the video.",
#         "A": "Identify the root cause or trigger of the traffic event shown.",
#         "F": "Predict the likely outcome based on vehicle trajectories and positions.",
#         "R": "Infer what events occurred before the current scene.",
#         "C": "Consider how the outcome would change under the hypothetical condition.",
#         "I": "Identify what action could have prevented the incident.",
#     }
    
#     type_key = question_type[0].upper() if question_type else "U"
#     hint = task_hints.get(type_key, task_hints["U"])
    
#     return (
#         f"Analyze this dashcam video and answer the question.\n\n"
#         f"Question Type: {question_type}\n"
#         f"Hint: {hint}\n\n"
#         f"Question: {question}\n\n"
#         f"Choices:\n{choices_text}\n\n"
#         f"Respond with ONLY the choice number (0-{num_choices - 1}).\n"
#         f"Answer:"
#     )


# def parse_model_answer(output: str, num_options: int = 4) -> int:
#     """Parse model output to extract answer index."""
#     output = output.strip()
    
#     # Try to find a single digit 0-3
#     for char in output:
#         if char.isdigit():
#             idx = int(char)
#             if 0 <= idx < num_options:
#                 return idx
    
#     # Fallback: check for letter answers (A=0, B=1, etc.)
#     output_upper = output.upper()
#     for i, letter in enumerate(['A', 'B', 'C', 'D']):
#         if letter in output_upper:
#             return i
    
#     return -1  # Could not parse


# # =============================================================================
# # RUN INFERENCE
# # =============================================================================

# results = []
# correct = 0
# total = 0

# for idx, item in enumerate(tqdm(test_data, desc="Processing videos")):
#     try:
#         vid_filename = item["vid_filename"]
#         video_path = os.path.join(video_base_path, vid_filename)
        
#         if not os.path.exists(video_path):
#             print(f"Warning: Video not found: {video_path}")
#             item["model_output"] = "ERROR: Video not found"
#             item["model_answer_idx"] = -1
#             item["is_correct"] = False
#             results.append(item)
#             continue
        
#         # Format prompt (matches training format)
#         prompt = format_mcq_prompt(item["question"], item["options"], item["question_type"])
        
#         # Run inference
#         with torch.inference_mode():
#             output, _ = model.chat(
#                 video_path=video_path,
#                 tokenizer=tokenizer,
#                 user_prompt=prompt,
#                 return_history=True,
#                 max_num_frames=max_num_frames,
#                 generation_config=generation_config
#             )
        
#         # Parse answer
#         predicted_idx = parse_model_answer(output, len(item["options"]))
        
#         item["model_output"] = output
#         item["model_answer_idx"] = predicted_idx
#         item["is_correct"] = (predicted_idx == item["answer"])
#         results.append(item)
        
#         total += 1
#         if item["is_correct"]:
#             correct += 1
        
#         if (idx + 1) % 50 == 0:
#             print(f"\nAccuracy at {idx + 1}: {correct}/{total} = {correct/total*100:.2f}%")
        
#         if (idx + 1) % 10 == 0:
#             torch.cuda.empty_cache()
        
#     except Exception as e:
#         print(f"\nError processing {item.get('vid_filename', 'unknown')}: {str(e)}")
#         item["model_output"] = f"ERROR: {str(e)}"
#         item["model_answer_idx"] = -1
#         item["is_correct"] = False
#         results.append(item)

# # =============================================================================
# # SAVE RESULTS
# # =============================================================================

# final_accuracy = correct / total * 100 if total > 0 else 0

# df_results = pd.DataFrame(results)
# df_results = df_results[["record_id", "vid_filename", "model_answer_idx", "answer", "is_correct"]]
# df_results.columns = ["id", "filename", "answer", "gt_answer", "is_correct"]
# df_results.to_csv(output_json_path, index=False)

# print(f"\n{'='*50}")
# print(f"Final Results:")
# print(f"Accuracy: {correct}/{total} = {final_accuracy:.2f}%")
# print(f"VRAM usage: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
# print(f"Results saved to: {output_json_path}")

In [ ]:
!python3 videochat_sutd.py

VIDEO_DIR: /kaggle/input/sutd-traffic-video-qa/videos_obj_tracking exists: True
QUESTIONS_DIR: /kaggle/input/sutd-traffic-video-qa/questions/questions exists: True
Found 6 question JSON file(s):
 - /kaggle/input/sutd-traffic-video-qa/questions/questions/R2_all.jsonl
 - /kaggle/input/sutd-traffic-video-qa/questions/questions/R2_test.jsonl
 - /kaggle/input/sutd-traffic-video-qa/questions/questions/R2_train.jsonl
 - /kaggle/input/sutd-traffic-video-qa/questions/questions/R3_all.jsonl
 - /kaggle/input/sutd-traffic-video-qa/questions/questions/R3_test.jsonl
 - /kaggle/input/sutd-traffic-video-qa/questions/questions/R3_train.jsonl
QUESTIONS_PATH: /kaggle/input/sutd-traffic-video-qa/questions/questions/R2_test.jsonl
OUTPUT_DIR: /kaggle/working/output
KEYFRAME_DIR: /kaggle/temp/keyframes
Loading model...
model.safetensors: 100%|████████████████████| 4.14G/4.14G [00:05<00:00, 812MB/s]
Loading vision tower: umt-hd-large
flash_v2
flash_v2
flash_v2
flash_v2
flash_v2
flash_v2
flash_v2
flash_v2
flas

In [ ]:
# Read answers\sutd_test_with_answers_video_chat_yolo_no_prompting.csv
import pandas as pd
df = pd.read_csv('./answers/sutd_test_with_answers_video_chat_yolo_no_prompting.csv')

# Display accuracy every 50 samples
total = len(df)
correct = 0
for i in range(total):
    if df.loc[i, 'answer'] == df.loc[i, 'gt_answer']:
        correct += 1
    if (i + 1) % 50 == 0:
        print(f"Accuracy at {i + 1}: {correct}/{i + 1} = {correct / (i + 1) * 100:.2f}%")


Accuracy at 50: 20/50 = 40.00%
Accuracy at 100: 42/100 = 42.00%
Accuracy at 150: 67/150 = 44.67%
Accuracy at 200: 93/200 = 46.50%
Accuracy at 250: 113/250 = 45.20%
Accuracy at 300: 134/300 = 44.67%
Accuracy at 350: 160/350 = 45.71%
Accuracy at 400: 182/400 = 45.50%
Accuracy at 450: 202/450 = 44.89%
Accuracy at 500: 221/500 = 44.20%
Accuracy at 550: 238/550 = 43.27%
Accuracy at 600: 265/600 = 44.17%
Accuracy at 650: 286/650 = 44.00%
Accuracy at 700: 310/700 = 44.29%
Accuracy at 750: 331/750 = 44.13%
Accuracy at 800: 355/800 = 44.38%
Accuracy at 850: 374/850 = 44.00%
Accuracy at 900: 399/900 = 44.33%
Accuracy at 950: 416/950 = 43.79%
Accuracy at 1000: 440/1000 = 44.00%
Accuracy at 1050: 461/1050 = 43.90%
Accuracy at 1100: 492/1100 = 44.73%
Accuracy at 1150: 514/1150 = 44.70%
Accuracy at 1200: 538/1200 = 44.83%
Accuracy at 1250: 554/1250 = 44.32%
Accuracy at 1300: 576/1300 = 44.31%
Accuracy at 1350: 600/1350 = 44.44%
Accuracy at 1400: 616/1400 = 44.00%
Accuracy at 1450: 644/1450 = 44.41%
